In [1]:
from config import Config
from DataManager import DataManager
from DataPreprocessor import Preprocessor
from FeatureSelection import FeatureSelector
from ModelManager import ModelManager
from ModelEvaluation import Evaluator

from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
cfg = Config()
data_manager = DataManager(cfg)
preprocessor = Preprocessor(cfg)

gt_df = data_manager.load_ground_truth()
wl = data_manager.load_wavelengths()

#image = data_manager.load_image(1200)
#processed = preprocessor.process_image(image)

X, y, valid_indices = preprocessor.process_dataset(data_manager, gt_df)
X.shape

(1732, 150)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=cfg.RANDOM_SEED)

In [5]:
selector = FeatureSelector(cfg, n_features=25)
selector.fit(X_train, y_train)

X_train_rfe = selector.transform(X_train)
X_test_rfe = selector.transform(X_test)

Fitting estimator with 150 features.
Fitting estimator with 145 features.
Fitting estimator with 140 features.
Fitting estimator with 135 features.
Fitting estimator with 130 features.
Fitting estimator with 125 features.
Fitting estimator with 120 features.
Fitting estimator with 115 features.
Fitting estimator with 110 features.
Fitting estimator with 105 features.
Fitting estimator with 100 features.
Fitting estimator with 95 features.
Fitting estimator with 90 features.
Fitting estimator with 85 features.
Fitting estimator with 80 features.
Fitting estimator with 75 features.
Fitting estimator with 70 features.
Fitting estimator with 65 features.
Fitting estimator with 60 features.
Fitting estimator with 55 features.
Fitting estimator with 50 features.
Fitting estimator with 45 features.
Fitting estimator with 40 features.
Fitting estimator with 35 features.
Fitting estimator with 30 features.


In [6]:
models = ModelManager(cfg)

models.tune_fit(X_train_rfe, y_train, verbose=True)
y_pred = models.predict(X_test_rfe)
y_baseline = models.get_baseline_predictions(X_train=X_train_rfe, y_train=y_train, X_test=X_test_rfe)


Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [P] parameter: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 20}

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [K] parameter: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 50}

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [Mg] parameter: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': None}

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [pH] parameter: {'n_estimators': 700, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 40}



In [22]:
evaluator = Evaluator(cfg)
results = evaluator.calculate_hyperview_score(y_test, y_pred, y_baseline)

print(results)
models.save_models(filename='best_rf_model.pkl')

P              0.971106
K              0.946097
Mg             0.905409
pH             0.844258
FINAL_SCORE    0.916717
dtype: float64
Successfully saved models to: ../Submissions\Models\best_rf_model.pkl!


In [16]:
models_new = ModelManager(cfg)
models_new.load_models()

y_pred_new = models_new.predict(X_test_rfe)
evaluator.calculate_hyperview_score(y_test, y_pred_new, y_baseline)

P              0.966009
K              0.945428
Mg             0.936602
pH             0.848595
FINAL_SCORE    0.924159
dtype: float64

In [14]:
#przykładowe do wywalenia później jak działa przykład jakby co to mogę to inaczej przerobić potem jakby to nie było to:)


import  numpy as np
import pandas as pd
from config import Config
from DataManager import DataManager
from DataPreprocessor import Preprocessor
from FeatureSelection import FeatureSelector
from KNN import KNNPhManager


cfg = Config()
dm = DataManager(cfg)
pp = Preprocessor(cfg)
gt_df = dm.load_ground_truth()

#micro trenig

# first 50 as   traing  base
train_subset_df = gt_df.iloc[:50]

# 51 as test
test_row = gt_df.iloc[50]
test_id = test_row['sample_index']
real_ph = test_row['pH']


print(f"Traingn with 50 images")
print(f"testing with image {test_id} (real pH: {real_ph})")

#processing of mtraing data

X_train, y_train, _ = pp.process_dataset(dm, train_subset_df)
X_train = np.array(X_train)
y_train = np.array(y_train)


# getting this single image
single_image = dm.load_image(int(test_id))
# to 1D
single_spectrum = pp.process_image(single_image)

# KNN needs  2D array [number of traing data, PH]
X_single_test = single_spectrum.reshape(1, -1)


fs = FeatureSelector(cfg, n_features=50)
fs.fit(X_train, y_train)

X_train_sel = fs.transform(X_train)
X_test_sel = fs.transform(X_single_test)


#all the working
knn = KNNPhManager(cfg)
knn.fit_tune(X_train_sel, y_train)
predicted_ph = knn.predict(X_test_sel)[0]

#

print(f"Test don for {test_id} image:")
print(f"Real pH: {real_ph:.4f}")
print(f"Predicted pH: {predicted_ph:.4f}")
print(f"Difference: {abs(real_ph - predicted_ph):.4f}")


Traingn with 50 images
testing with image 50.0 (real pH: 7.0)
Fitting estimator with 150 features.
Fitting estimator with 145 features.
Fitting estimator with 140 features.
Fitting estimator with 135 features.
Fitting estimator with 130 features.
Fitting estimator with 125 features.
Fitting estimator with 120 features.
Fitting estimator with 115 features.
Fitting estimator with 110 features.
Fitting estimator with 105 features.
Fitting estimator with 100 features.
Fitting estimator with 95 features.
Fitting estimator with 90 features.
Fitting estimator with 85 features.
Fitting estimator with 80 features.
Fitting estimator with 75 features.
Fitting estimator with 70 features.
Fitting estimator with 65 features.
Fitting estimator with 60 features.
Fitting estimator with 55 features.
Fitting 3 folds for each of 14 candidates, totalling 42 fits
Best  KNN parameters for pH: {'knn__metric': 'euclidean', 'knn__n_neighbors': 5, 'knn__weights': 'distance'}
Best score (MSE) : 0.0529
Test don fo